In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import io
import matplotlib.patches as mpatches
from google.colab import files

print("Selecione os arquivos CSV gerados (mqttsn.csv e p4ssn.csv):")
uploaded = files.upload()

dfs = []
for filename in uploaded.keys():
    df_temp = pd.read_csv(io.BytesIO(uploaded[filename]))
    df_temp['cenario'] = df_temp['cenario'].replace({'mqttsn': 'MQTT-SN', 'p4ssn': 'P4SSN'}).str.upper()
    dfs.append(df_temp)

df_completo = pd.concat(dfs, ignore_index=True)

def format_qos(row):
    # O valor 3 no CSV representa o QoS -1
    if row['qos'] == 3:
        return "QoS -1"
    
    # Aplica o sufixo de Retain para qualquer QoS (0, 1 ou 2)
    label = f"QoS {int(row['qos'])}"
    label += " (Com Retain)" if row['retain'] == 1 else ""
    return label

df_completo['QoS_Config'] = df_completo.apply(format_qos, axis=1)

# Médias para o gráfico
df_plot = df_completo.groupby(['cenario', 'QoS_Config'])['t_flow_ms'].mean().reset_index()
df_plot.rename(columns={'cenario': 'Topologia', 't_flow_ms': 'Tempo Médio do Fluxo (ms)'}, inplace=True)

sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 7))

# Ordem do eixo X - Atualizada para incluir possíveis variações de retain nos outros níveis
ordem_x = [
    "QoS -1", 
    "QoS 0", "QoS 0 (Com Retain)", 
    "QoS 1", "QoS 1 (Com Retain)",
    "QoS 2", "QoS 2 (Com Retain)"
]
palette_colors = sns.color_palette("viridis", 2)

ordem_existente = [o for o in ordem_x if o in df_plot['QoS_Config'].unique()]

ax = sns.barplot(
    data=df_plot,
    x='QoS_Config',
    y='Tempo Médio do Fluxo (ms)',
    hue='Topologia',
    palette=palette_colors,
    order=ordem_existente
)

# Hachuras e Legenda Customizada
patterns = ['...', '///']
topologias = sorted(df_plot['Topologia'].unique())

# Ajuste automático das hachuras baseado na quantidade de barras por categoria
for i, patch in enumerate(ax.patches):
    if i < len(ordem_existente):
        patch.set_hatch(patterns[0])
    else:
        patch.set_hatch(patterns[1])

patch1 = mpatches.Patch(facecolor=palette_colors[0], hatch=patterns[0], label=topologias[0])
patch2 = mpatches.Patch(facecolor=palette_colors[1], hatch=patterns[1], label=topologias[1])

plt.legend(handles=[patch1, patch2], title='Protocolo', loc='upper left', frameon=True)

plt.title('MQTT-SN x P4SSN', fontsize=15, pad=20)
plt.xlabel('Nível de QoS', fontsize=12)
plt.ylabel('Tempo Médio do Fluxo de Publicação (ms)', fontsize=12)

plt.tight_layout()

#plt.savefig("grafico_benchmark1.eps", format='eps', bbox_inches='tight')

plt.show()